# SC-Total TFT 다중 호라이즌(t+1~t+8) 예측 PoC

- Spec: `../specs/train-sc-tft-multistep.md`
- 인터뷰 spec: `../.omc/specs/deep-interview-tft-poc.md`
- 목표: SC-Total 주간 판매량을 t+1 ~ t+8 동시 예측 (TFT)
- 베이스라인: LightGBM (t+1) + Naive seasonal (t+1..t+8)
- 산출물: 본 노트북 셀 출력 + MLflow `http://10.90.8.125:5000/` artifact (experiment `SC_Total_TFT`)

**셀 표기**: 결정적 셀은 그대로 실행, `# TODO(domain): ...` 표시는 사용자 도메인 결정이 필요한 부분.

## 1. 셋업
환경 변수, MLflow URI, 시드, 디바이스(MPS/CUDA/CPU) 결정.

In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import mlflow

MLFLOW_TRACKING_URI = "http://10.90.8.125:5000/"
MLFLOW_EXPERIMENT = "SC_Total_TFT"
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"device={DEVICE}, torch={torch.__version__}, mlflow={mlflow.__version__}")

## 2. 데이터 로드 (Snowflake → parquet 캐시)

결정적: parquet 캐시가 있으면 즉시 로드.<br/>
도메인 결정: Snowflake 추출 코드(연결 정보, 쿼리, 컬럼 셀렉트). 1회 실행 후 캐시되면 이후 재실행은 빠름.

참고 SSOT: `serp-distSupplementAI/src/worker/data/...` (SC-Total 빌더).

In [ ]:
from pathlib import Path

DATA_DIR = Path("../data")
PARQUET_PATH = DATA_DIR / "sc_weekly.parquet"
RAW_TFT_CSV = DATA_DIR / "raw_tft.csv"
LEGACY_CSV = DATA_DIR / "sc_weekly.csv"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 우선순위: parquet 캐시 → raw_tft.csv → sc_weekly.csv (legacy)
if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    if 'WEEK_START' in df.columns:
        df['WEEK_START'] = pd.to_datetime(df['WEEK_START'])
    print(f"[parquet 캐시 로드] {PARQUET_PATH}  shape={df.shape}")
else:
    src_csv = RAW_TFT_CSV if RAW_TFT_CSV.exists() else (LEGACY_CSV if LEGACY_CSV.exists() else None)
    if src_csv is None:
        raise FileNotFoundError(
            f"입력 데이터 없음. data/ 폴더에 다음 중 하나를 두세요:\n"
            f"  - {PARQUET_PATH}\n"
            f"  - {RAW_TFT_CSV}\n"
            f"  - {LEGACY_CSV}\n"
            f"\n"
            f"추출: DataGrip에서 sql/extract_sc_weekly.sql 실행 → Fetch all → Export CSV → 위 경로 저장"
        )
    print(f"[CSV → parquet 변환] {src_csv}")
    df = pd.read_csv(src_csv)
    for c in ('WEEK_START', 'SSN_START', 'SSN_END'):
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce')
    df.to_parquet(PARQUET_PATH, index=False)
    print(f"  → parquet 캐시 생성: {PARQUET_PATH}  shape={df.shape}")

# 핵심 컬럼 점검 (운영 SSOT 기준)
expected = {'SC_CD', 'WEEK_START', 'WEEKLY_SALE_QTY'}
missing = expected - set(df.columns)
if missing:
    print(f"⚠️  핵심 컬럼 누락: {missing}  → 셀 4의 컬럼명을 추출본에 맞춰 조정 필요")

print(f"\n컬럼 ({len(df.columns)}개): {list(df.columns)}")
print(f"\ndtypes:\n{df.dtypes}")
df.head()

## 3. EDA
결정적: 기간/SC 수/결측/SC당 관측 주차 분포/lag autocorr → encoder length(52 vs 104) 결정 근거.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 운영 SSOT 컬럼명 (raw_tft.csv 기준) — 이후 셀들도 이 변수를 사용
GROUP_KEY = 'SC_CD'
TARGET = 'WEEKLY_SALE_QTY'

print(f"기간: {df['WEEK_START'].min().date()} ~ {df['WEEK_START'].max().date()}")
print(f"고유 {GROUP_KEY} 수: {df[GROUP_KEY].nunique():,}")
print(f"행 수: {len(df):,}")

weeks_per_sc = df.groupby(GROUP_KEY).size()
print(f"\n{GROUP_KEY}당 관측 주차 분포:\n{weeks_per_sc.describe().round(1)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(weeks_per_sc, bins=30); axes[0].set_title(f'{GROUP_KEY}당 관측 주차 수'); axes[0].set_xlabel('주차'); axes[0].set_ylabel(f'{GROUP_KEY} 수')
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].plot.bar(ax=axes[1], title='컬럼별 결측 비율')
plt.tight_layout(); plt.show()

# 자기상관(numpy 기반 ACF)
def acf_numpy(x, nlags):
    x = np.asarray(x, dtype=float); x = x - x.mean()
    var = (x ** 2).mean()
    if var == 0: return [1.0] + [0.0] * nlags
    return [1.0] + [float((x[:-i] * x[i:]).mean() / var) for i in range(1, nlags + 1)]

sample_sc = weeks_per_sc[weeks_per_sc >= 100].index[0] if (weeks_per_sc >= 100).any() else weeks_per_sc.idxmax()
sample_y = df[df[GROUP_KEY] == sample_sc].sort_values('WEEK_START')[TARGET].values
nlag = min(60, len(sample_y) // 2)
ac = acf_numpy(sample_y, nlag)
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(nlag + 1), ac); ax.axhline(0, color='k', lw=0.5)
ax.axvline(52, color='r', ls='--', alpha=0.5, label='1년')
ax.set_title(f'자기상관 (sample {GROUP_KEY}={sample_sc}) — peak이 52에 있으면 encoder_len=52 권장')
ax.set_xlabel('lag (주)'); ax.set_ylabel('ACF'); ax.legend()
plt.tight_layout(); plt.show()

# === A. cutoff별 활성 SC들의 history 길이 (warm cutoff 후보 추출) ===
candidate_cutoffs = pd.date_range(
    df['WEEK_START'].min() + pd.Timedelta(weeks=4),
    df['WEEK_START'].max() - pd.Timedelta(weeks=8),
    freq='4W-MON',
)
cutoff_stats = []
for c in candidate_cutoffs:
    active_sc = df[df['WEEK_START'] <= c].groupby(GROUP_KEY).size()
    if len(active_sc) == 0: continue
    cutoff_stats.append({
        'cutoff': c, 'n_sc_active': len(active_sc),
        'mean_history': active_sc.mean(),
        'median_history': active_sc.median(),
        'p25_history': active_sc.quantile(0.25),
    })
cutoff_stats_df = pd.DataFrame(cutoff_stats)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cutoff_stats_df['cutoff'], cutoff_stats_df['mean_history'], marker='o', label='mean')
ax.plot(cutoff_stats_df['cutoff'], cutoff_stats_df['median_history'], marker='s', label='median')
ax.plot(cutoff_stats_df['cutoff'], cutoff_stats_df['p25_history'], marker='^', label='p25')
ax.axhline(13, color='r', ls='--', alpha=0.5, label='cold 임계')
ax.axhline(52, color='g', ls='--', alpha=0.5, label='warm 임계')
ax.set_title('cutoff별 활성 SC들의 history 길이')
ax.set_xlabel('cutoff_date'); ax.set_ylabel('주'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# === B. SESN_SUB별 시즌 시작 추출 (cold cutoff 후보) ===
# F&F 시즌 코드: 1=Spring, 2=Pre-Summer, 3=Summer, 4=Fall, 5=Pre-Winter, 6=Winter
SESN_NAMES = {1: 'Spring', 2: 'Pre-Summer', 3: 'Summer', 4: 'Fall', 5: 'Pre-Winter', 6: 'Winter'}

cold_candidates = pd.DataFrame()
if 'SESN_SUB' in df.columns:
    first_sale = (df.groupby(GROUP_KEY)
                  .agg(first_week=('WEEK_START', 'min'), sesn=('SESN_SUB', 'first'))
                  .reset_index())
    first_sale['sesn_int'] = pd.to_numeric(first_sale['sesn'], errors='coerce').astype('Int64')
    first_sale['sesn_name'] = first_sale['sesn_int'].map(SESN_NAMES)
    first_sale['year'] = first_sale['first_week'].dt.year

    season_starts = (first_sale.groupby(['year', 'sesn_int', 'sesn_name'])['first_week']
                     .agg(['min', 'median', 'count']).reset_index()
                     .rename(columns={'min': 'earliest', 'median': 'typical_start', 'count': 'n_sc'}))
    season_starts['cold_cutoff'] = season_starts['typical_start'] + pd.Timedelta(weeks=2)
    cold_candidates = (season_starts[
        (season_starts['cold_cutoff'] < df['WEEK_START'].max() - pd.Timedelta(weeks=8))
        & (season_starts['cold_cutoff'] > df['WEEK_START'].min() + pd.Timedelta(weeks=8))
    ].sort_values('cold_cutoff'))

    print('=== (year, SESN_SUB)별 시즌 시작 추정 (typical_start = SC 첫출시 주차의 중위) ===')
    print(season_starts.tail(12).to_string(index=False))
    print('\n=== cold cutoff 후보 (시즌 시작 + 2주) ===')
    print(cold_candidates[['year', 'sesn_int', 'sesn_name', 'typical_start', 'cold_cutoff', 'n_sc']].tail(8).to_string(index=False))
    print('\n→ 셀 5 CUTOFFS 패턴: cold = cold_candidates.tail(2)["cold_cutoff"].tolist()')
else:
    print('⚠️  SESN_SUB 컬럼 없음 — 시즌 단계 분석 스킵')

## 4. 피처 정의 + TimeSeriesDataSet
결정적: time_idx 생성, dtype 정리, train/val 분할, `pytorch_forecasting.TimeSeriesDataSet` 생성.<br/>
도메인 결정: encoder_len(52 vs 104), 컬럼 분류가 실제 df 컬럼과 맞는지 검증.

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import EncoderNormalizer

ENCODER_LEN = 52       # max: 베테랑 SC 1년 lookback
MIN_ENCODER_LEN = 4    # min: 시즌 초기 cold-start (4주 history 필수)
DECODER_LEN = 8

GROUP_IDS = [GROUP_KEY]  # ['SC_CD']
STATIC_CATEGORICALS = ['SEX', 'FIT_INFO1', 'FAB_TYPE', 'ITEM', 'SESN_SUB',
                       'PRDT_KIND_CD', 'BRAND_CD', 'SSN_CD']
TIME_VARYING_KNOWN_REALS = ['WEEK_OF_YEAR', 'TAG_PRICE',
                            'FCST_AVG_MIN_TEMP', 'FCST_AVG_MAX_TEMP', 'FCST_TOTAL_PCP']
TIME_VARYING_UNKNOWN_REALS = ['WEEKLY_DISC_RAT', 'BOW_STOCK', 'STOCK_RATIO',
                              'CUM_INTAKE', 'SCS_W_DISC_RATE', 'NUM_SIZES']

# WEEK_OF_YEAR 자체 계산
df['WEEK_OF_YEAR'] = df['WEEK_START'].dt.isocalendar().week.astype(int)

# === 시즌 기간 필터 (이월 상품 분포 제거) ===
# 사용자 비즈니스 시나리오 = 새 시즌 신상품 예측 → 시즌 종료 후 이월 판매는 학습/평가 대상 X
if 'SSN_START' in df.columns and 'SSN_END' in df.columns:
    n_before = len(df)
    df = df[(df['WEEK_START'] >= df['SSN_START']) & (df['WEEK_START'] <= df['SSN_END'])].copy()
    print(f'SSN 기간 필터: {n_before:,} → {len(df):,} 행 ({len(df)/n_before:.1%} 유지)')

# time_idx (필터 후 SC별 주차 순서 재계산)
df = df.sort_values([GROUP_KEY, 'WEEK_START']).reset_index(drop=True)
df['time_idx'] = df.groupby(GROUP_KEY).cumcount()

# dtype 정리
for c in STATIC_CATEGORICALS:
    if c in df.columns:
        df[c] = df[c].fillna('__missing__').astype(str)

real_cols = [c for c in TIME_VARYING_KNOWN_REALS + TIME_VARYING_UNKNOWN_REALS + [TARGET] if c in df.columns]
df[real_cols] = df[real_cols].astype(float).fillna(0.0)

# 누락 컬럼 점검
missing_cols = [c for c in STATIC_CATEGORICALS + TIME_VARYING_KNOWN_REALS + TIME_VARYING_UNKNOWN_REALS
                if c not in df.columns]
if missing_cols:
    print(f'⚠️  df에 없는 컬럼: {missing_cols}')

STATIC_CATEGORICALS = [c for c in STATIC_CATEGORICALS if c in df.columns]
TIME_VARYING_KNOWN_REALS = [c for c in TIME_VARYING_KNOWN_REALS if c in df.columns]
TIME_VARYING_UNKNOWN_REALS = [c for c in TIME_VARYING_UNKNOWN_REALS if c in df.columns]

max_idx = int(df['time_idx'].max())
TRAINING_CUTOFF = max_idx - DECODER_LEN

training = TimeSeriesDataSet(
    df[df['time_idx'] <= TRAINING_CUTOFF],
    time_idx='time_idx',
    target=TARGET,
    group_ids=GROUP_IDS,
    min_encoder_length=MIN_ENCODER_LEN,
    max_encoder_length=ENCODER_LEN,
    min_prediction_length=1,
    max_prediction_length=DECODER_LEN,
    static_categoricals=STATIC_CATEGORICALS,
    time_varying_known_reals=['time_idx'] + TIME_VARYING_KNOWN_REALS,
    time_varying_unknown_reals=[TARGET] + TIME_VARYING_UNKNOWN_REALS,
    # ⭐ 신상품 SC도 정규화 가능하게 EncoderNormalizer (sample 단위 정규화)
    target_normalizer=EncoderNormalizer(transformation='softplus'),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)
validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)
print(f'\ntraining samples={len(training):,}  validation samples={len(validation):,}')
print(f'  ENCODER_LEN={ENCODER_LEN} (min {MIN_ENCODER_LEN})  DECODER_LEN={DECODER_LEN}')
print(f'  STATIC_CATEGORICALS ({len(STATIC_CATEGORICALS)}): {STATIC_CATEGORICALS}')
print(f'  TIME_VARYING_KNOWN_REALS ({len(TIME_VARYING_KNOWN_REALS)}): {TIME_VARYING_KNOWN_REALS}')
print(f'  TIME_VARYING_UNKNOWN_REALS ({len(TIME_VARYING_UNKNOWN_REALS)}): {TIME_VARYING_UNKNOWN_REALS}')

## 5. Naive seasonal baseline (t+1..t+8)
결정적: WAPE 함수, walk-forward cutoff 4개, 두 종류 naive(`lag-52`, `WoY 다년 평균`) horizon별 평가, 더 보수적인 쪽을 baseline floor로 채택.<br/>
도메인 결정: CUTOFFS 주차 셋 (default = 마지막 8주 직전 4주).

In [ ]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float); y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true).sum()
    return float(np.abs(y_true - y_pred).sum() / denom) if denom > 0 else float('nan')

# === Walk-forward CUTOFFS — 시즌 단계별 평가 활성화 ===
LATEST_WEEK = df['WEEK_START'].max()

# default fallback: 단순 최근 4주
CUTOFFS = sorted([LATEST_WEEK - pd.Timedelta(weeks=DECODER_LEN + k) for k in range(4)])

# 권장 패턴: cold(시즌 시작 직후) 2개 + warm(시즌 중반·말) 2개
if 'cold_candidates' in globals() and len(cold_candidates) >= 2 and 'cutoff_stats_df' in globals():
    cold = cold_candidates.tail(2)['cold_cutoff'].tolist()
    warm = cutoff_stats_df.nlargest(2, 'median_history')['cutoff'].tolist()
    CUTOFFS = sorted(set(cold + warm))
    print('CUTOFFS = cold(2) + warm(2) — 시즌 단계별 평가')
else:
    print('⚠️  cold_candidates / cutoff_stats_df 없음 — 단순 최근 4주 fallback')

print(f'CUTOFFS: {[c.strftime("%Y-%m-%d") for c in CUTOFFS]}')

def _recent_mean(d, sc, cutoff, n=4):
    """시즌 초기 fallback: cutoff 이전 직전 n주 평균."""
    recent = d[(d[GROUP_KEY] == sc) & (d['WEEK_START'] <= cutoff)].sort_values('WEEK_START').tail(n)
    return float(recent[TARGET].mean()) if len(recent) else np.nan

def naive_lag52(d, sc, cutoff, h):
    target_w = cutoff + pd.Timedelta(weeks=h)
    src_w = target_w - pd.Timedelta(weeks=52)
    sub = d[(d[GROUP_KEY] == sc) & (d['WEEK_START'] == src_w)]
    if len(sub):
        return float(sub[TARGET].iloc[0])
    return _recent_mean(d, sc, cutoff)  # 시즌 초기

def naive_woy_mean(d, sc, cutoff, h):
    target_w = cutoff + pd.Timedelta(weeks=h)
    target_woy = int(target_w.isocalendar().week)
    sub = d[(d[GROUP_KEY] == sc) & (d['WEEK_START'] <= cutoff) & (d['WEEK_OF_YEAR'] == target_woy)]
    if len(sub):
        return float(sub[TARGET].mean())
    return _recent_mean(d, sc, cutoff)  # 시즌 초기

def naive_eval(predict_fn):
    rows = []
    for cutoff in CUTOFFS:
        for h in range(1, DECODER_LEN + 1):
            tw = cutoff + pd.Timedelta(weeks=h)
            actual = df[df['WEEK_START'] == tw].set_index(GROUP_KEY)[TARGET]
            pred = pd.Series({sc: predict_fn(df, sc, cutoff, h) for sc in actual.index})
            mask = pred.notna() & actual.notna()
            if mask.sum():
                rows.append({'cutoff': cutoff, 'h': h, 'wape': wape(actual[mask], pred[mask])})
    return pd.DataFrame(rows)

naive_a = naive_eval(naive_lag52)
naive_b = naive_eval(naive_woy_mean)

naive_summary = pd.concat([
    naive_a.groupby('h')['wape'].mean().rename('lag52'),
    naive_b.groupby('h')['wape'].mean().rename('woy_mean'),
], axis=1)
naive_summary['baseline'] = naive_summary.min(axis=1)
print('\n=== Naive baselines (CUTOFFS 평균 WAPE) ===')
print(naive_summary.round(4).to_string())

# cutoff별 분리 (시즌 단계별 평가용)
naive_per_cutoff = (pd.concat([naive_a.assign(name='lag52'), naive_b.assign(name='woy_mean')])
                   .groupby(['cutoff', 'h', 'name'])['wape'].mean().unstack('name'))
naive_per_cutoff['baseline'] = naive_per_cutoff.min(axis=1)
print('\n=== Naive baseline (cutoff별, baseline=min(lag52, woy_mean)) ===')
print(naive_per_cutoff['baseline'].unstack('h').round(4).to_markdown())

## 6. LightGBM 베이스라인 회수
**Option A (권장)**: 사내 MLflow `SC_Total_LightGBM` run에서 동일 cutoff의 t+1 prediction artifact 다운로드 — 운영 모델과 정확히 head-to-head.<br/>
**Option B (폴백)**: 본 셀에서 lag_1 + WEEK_OF_YEAR + 범주형만 쓰는 미니 LightGBM 재학습 — 운영 모델보다 약하므로 비교 정확도가 낮음.

도메인 결정: A vs B 선택, A이면 cutoff→run_id 매핑.

In [ ]:
import lightgbm as lgb

USE_OPTION_A = False  # TODO(domain): 사내 MLflow에 cutoff별 prediction이 있으면 True

if USE_OPTION_A:
    from mlflow.tracking import MlflowClient
    client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
    # TODO(domain): cutoff_date tag로 run 매칭 → predictions.parquet 다운로드
    raise NotImplementedError('Option A 매핑 코드를 채우세요')
else:
    def mini_lgb_t1(d, cutoff):
        train = d[d['WEEK_START'] <= cutoff].sort_values([GROUP_KEY, 'WEEK_START']).copy()
        train['lag_1'] = train.groupby(GROUP_KEY)[TARGET].shift(1)
        last_y = train.groupby(GROUP_KEY)[TARGET].last().rename('lag_1')
        target_w = cutoff + pd.Timedelta(weeks=1)
        test = d[d['WEEK_START'] == target_w].set_index(GROUP_KEY).join(last_y, how='left')
        feats = ['lag_1', 'WEEK_OF_YEAR'] + STATIC_CATEGORICALS
        # 신상품(lag_1=NaN)도 평가에 포함 — LightGBM 자체 NaN-handling
        required = ['WEEK_OF_YEAR'] + STATIC_CATEGORICALS
        train_clean = train.dropna(subset=required + [TARGET])
        test_clean = test.dropna(subset=required)
        if len(test_clean) == 0:
            return pd.Series(dtype=float)
        for c in STATIC_CATEGORICALS:
            train_clean[c] = train_clean[c].astype('category')
            test_clean[c] = test_clean[c].astype('category')
        model = lgb.LGBMRegressor(objective='poisson', n_estimators=300, learning_rate=0.05, verbose=-1)
        model.fit(train_clean[feats], train_clean[TARGET], categorical_feature=STATIC_CATEGORICALS)
        return pd.Series(model.predict(test_clean[feats]).clip(min=0), index=test_clean.index)

    lgb_rows = []
    for cutoff in CUTOFFS:
        pred = mini_lgb_t1(df, cutoff)
        target_w = cutoff + pd.Timedelta(weeks=1)
        actual = df[df['WEEK_START'] == target_w].set_index(GROUP_KEY)[TARGET]
        common = pred.index.intersection(actual.index)
        if len(common):
            lgb_rows.append({'cutoff': cutoff, 'h': 1, 'wape_lgb': wape(actual.loc[common], pred.loc[common])})
    lgb_df = pd.DataFrame(lgb_rows)
    print('[Mini LightGBM t+1] cutoff별 WAPE:')
    print(lgb_df.to_string(index=False))
    print(f'\n평균 t+1 WAPE = {lgb_df["wape_lgb"].mean():.4f}')

## 7. TFT 학습
결정적: pytorch-forecasting TFT, MLflowLogger, EarlyStopping/Checkpoint 보일러플레이트.<br/>
도메인 결정: hidden_size, attention_head_size, dropout, learning_rate, max_epochs (default는 R&D 표준 시작값).

In [ ]:
import pytorch_lightning as pl
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

BATCH_SIZE = 128
MAX_EPOCHS = 30          # TODO(domain): 학습 시간 보고 조정
HIDDEN_SIZE = 64         # TODO(domain): 32~128 튜닝
DROPOUT = 0.1

train_dl = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dl = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=4,
    dropout=DROPOUT,
    hidden_continuous_size=min(32, HIDDEN_SIZE),
    loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
    log_interval=10,
    reduce_on_plateau_patience=4,
)
print(f'TFT params = {sum(p.numel() for p in tft.parameters()):,}')

mlflow_logger = MLFlowLogger(
    experiment_name=MLFLOW_EXPERIMENT,
    tracking_uri=MLFLOW_TRACKING_URI,
)
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    gradient_clip_val=0.1,
    logger=mlflow_logger,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, mode='min'),
        ModelCheckpoint(monitor='val_loss', save_top_k=1, mode='min', filename='tft-{epoch:02d}-{val_loss:.4f}'),
    ],
    enable_progress_bar=True,
)
trainer.fit(tft, train_dataloaders=train_dl, val_dataloaders=val_dl)
best_path = trainer.checkpoint_callback.best_model_path
print(f'best ckpt: {best_path}')

## 8. 평가 — horizon별 WAPE 표
결정적: best ckpt 로드, validation set 예측(P50), horizon별 WAPE 계산, TFT vs LightGBM(t+1) vs Naive 비교 표 + 그림 + MLflow 등록.

노트: PoC에서는 단일 학습 + validation 마지막 8주를 multi-horizon 평가로 사용 (cutoff별 재학습은 비용 큼). 진정한 walk-forward 학습은 추후 단계.

In [ ]:
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_path)

# === A. validation set 단일 평균 평가 ===
predictions = best_tft.predict(val_dl, mode='quantiles', return_x=True, return_y=True)
y_true = predictions.y[0].cpu().numpy() if hasattr(predictions.y, '__len__') else predictions.y.cpu().numpy()
y_pred = predictions.output.cpu().numpy()
median_idx = 1  # quantiles=[0.1, 0.5, 0.9]
y_p50 = y_pred[..., median_idx]

tft_h = pd.DataFrame([{'h': h+1, 'wape_tft': wape(y_true[:, h], y_p50[:, h])} for h in range(DECODER_LEN)]).set_index('h')

compare = naive_summary[['baseline']].rename(columns={'baseline': 'wape_naive'}).join(tft_h)
if 'lgb_df' in globals() and len(lgb_df):
    compare = compare.join(lgb_df.groupby('h')['wape_lgb'].mean(), how='left')
print('=== validation 평균 horizon WAPE ===')
print(compare.round(4).to_markdown())

with mlflow.start_run(run_name='tft_eval_summary'):
    mlflow.log_dict(compare.reset_index().round(6).to_dict(orient='records'), 'wape_by_horizon.json')
    fig, ax = plt.subplots(figsize=(8, 4))
    compare.plot(ax=ax, marker='o')
    ax.set_title('horizon별 WAPE — TFT vs LightGBM(t+1) vs Naive')
    ax.set_xlabel('horizon h (주)'); ax.set_ylabel('WAPE'); ax.grid(alpha=0.3)
    mlflow.log_figure(fig, 'wape_by_horizon.png')
    plt.show()

# === B. cutoff(시즌 단계)별 분리 평가 + breakdown 헬퍼 ===
def tft_predict_at_cutoff(model, df_full, cutoff, base_dataset, decoder_len):
    """학습된 TFT로 특정 cutoff 시점의 multi-horizon 예측."""
    cutoff_idx = int(df_full.loc[df_full['WEEK_START'] <= cutoff, 'time_idx'].max())
    sub = df_full[df_full['time_idx'] <= cutoff_idx + decoder_len].copy()
    ds = TimeSeriesDataSet.from_dataset(base_dataset, sub, predict=True, stop_randomization=True)
    dl = ds.to_dataloader(train=False, batch_size=128, num_workers=0)
    return model.predict(dl, mode='quantiles', return_x=True, return_y=True, return_index=True)

cutoff_predictions = {}
cutoff_h_rows = []
for cutoff in CUTOFFS:
    try:
        preds_c = tft_predict_at_cutoff(best_tft, df, cutoff, training, DECODER_LEN)
        cutoff_predictions[cutoff] = preds_c
        y_t = preds_c.y[0].cpu().numpy() if hasattr(preds_c.y, '__len__') else preds_c.y.cpu().numpy()
        y_p = preds_c.output.cpu().numpy()[..., median_idx]
        for h in range(DECODER_LEN):
            cutoff_h_rows.append({
                'cutoff': cutoff.strftime('%Y-%m-%d'), 'h': h+1, 'n': int(len(y_t)),
                'wape_tft': wape(y_t[:, h], y_p[:, h]),
            })
    except Exception as e:
        print(f'cutoff {cutoff.date()} TFT 예측 실패: {e}')

cutoff_h_df = pd.DataFrame(cutoff_h_rows)
if len(cutoff_h_df):
    tft_pivot = cutoff_h_df.pivot_table(index='cutoff', columns='h', values='wape_tft')
    naive_pivot = naive_per_cutoff['baseline'].unstack('h')
    naive_pivot.index = [c.strftime('%Y-%m-%d') if hasattr(c, 'strftime') else str(c) for c in naive_pivot.index]
    print('\n=== TFT WAPE × cutoff (시즌 단계) × horizon ===')
    print(tft_pivot.round(4).to_markdown())
    print('\n=== Naive baseline × cutoff × horizon ===')
    print(naive_pivot.round(4).to_markdown())
    print('\n=== TFT − Naive (음수 = TFT 우월) ===')
    diff = (tft_pivot - naive_pivot).round(4)
    print(diff.to_markdown())

# === C. PRDT_KIND_CD × horizon (의류 카테고리별 차별화) ===
# === D. SC history 길이 bin × horizon (cold-start 차별화) ===
HIST_BINS_EVAL = [0, 4, 13, 26, 999]
HIST_LABELS_EVAL = ['ULTRA(<4)', 'COLD(4-12)', 'WARM(13-25)', 'MATURE(26+)']
sc_to_kind = df.groupby('SC_CD')['PRDT_KIND_CD'].first()

prdt_h_rows = []
hist_h_rows = []
for cutoff, preds_c in cutoff_predictions.items():
    try:
        # return_index=True 시 preds_c.index가 DataFrame (group_ids + time_idx)
        idx_df = preds_c.index
        if 'SC_CD' not in idx_df.columns:
            continue
        sc_ids = idx_df['SC_CD'].astype(str).values
    except Exception as e:
        print(f'cutoff {cutoff.date()} index 추출 실패: {e}')
        continue

    y_t = preds_c.y[0].cpu().numpy() if hasattr(preds_c.y, '__len__') else preds_c.y.cpu().numpy()
    y_p = preds_c.output.cpu().numpy()[..., median_idx]

    # PRDT_KIND_CD 매핑
    sample_kind = pd.Series(sc_ids).map(sc_to_kind).fillna('UNK')
    for kind in sample_kind.unique():
        mask = (sample_kind == kind).values
        if mask.sum() < 5:
            continue
        for h in range(DECODER_LEN):
            prdt_h_rows.append({
                'PRDT_KIND_CD': kind, 'cutoff': cutoff.strftime('%Y-%m-%d'),
                'h': h+1, 'n': int(mask.sum()),
                'wape_tft': wape(y_t[mask, h], y_p[mask, h]),
            })

    # SC history 길이 bin (cutoff 시점 기준)
    sc_hist = df[df['WEEK_START'] <= cutoff].groupby('SC_CD').size()
    sample_hist = pd.Series(sc_ids).map(sc_hist).fillna(0).astype(int)
    sample_bin = pd.cut(sample_hist, bins=HIST_BINS_EVAL, labels=HIST_LABELS_EVAL, right=False)
    for bin_label in HIST_LABELS_EVAL:
        mask = (sample_bin == bin_label).values
        if mask.sum() < 5:
            continue
        for h in range(DECODER_LEN):
            hist_h_rows.append({
                'history_bin': bin_label, 'cutoff': cutoff.strftime('%Y-%m-%d'),
                'h': h+1, 'n': int(mask.sum()),
                'wape_tft': wape(y_t[mask, h], y_p[mask, h]),
            })

prdt_h_df = pd.DataFrame(prdt_h_rows)
hist_h_df = pd.DataFrame(hist_h_rows)

if len(prdt_h_df):
    print('\n=== TFT WAPE × PRDT_KIND_CD × horizon (의류 카테고리별 차별화) ===')
    print(prdt_h_df.groupby(['PRDT_KIND_CD', 'h'])['wape_tft'].mean().unstack('h').round(4).to_markdown())

if len(hist_h_df):
    print('\n=== TFT WAPE × history bin × horizon (cold-start 차별화 — 비즈니스 임팩트 영역) ===')
    print(hist_h_df.groupby(['history_bin', 'h'])['wape_tft'].mean().unstack('h').round(4).to_markdown())

# === E. Multi-horizon 일관성 검증 (TFT의 native multi-horizon 가치 입증) ===
# t+1..t+8 P50 시계열의 인접 horizon 간 |Δ| 분포 → smoothness 검증
adj_diffs = np.abs(np.diff(y_p50, axis=1))  # (n_samples, decoder_len-1)
print(f'\n=== Multi-horizon 일관성 (인접 horizon 간 |Δ y_p50|) ===')
print(f'mean |Δ|: {adj_diffs.mean():.3f}  median: {np.median(adj_diffs):.3f}  p95: {np.percentile(adj_diffs, 95):.3f}')

# MLflow 종합 등록
with mlflow.start_run(run_name='tft_eval_breakdowns'):
    if len(cutoff_h_df):
        mlflow.log_dict(cutoff_h_df.to_dict(orient='records'), 'wape_by_cutoff_horizon.json')
    if len(prdt_h_df):
        mlflow.log_dict(prdt_h_df.to_dict(orient='records'), 'wape_by_prdt_kind_horizon.json')
    if len(hist_h_df):
        mlflow.log_dict(hist_h_df.to_dict(orient='records'), 'wape_by_history_bin_horizon.json')
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(adj_diffs.flatten(), bins=50)
    ax.set_title('Multi-horizon 일관성 — 인접 horizon 간 |Δ y_p50| 분포')
    ax.set_xlabel('|Δ|'); ax.set_ylabel('count'); ax.grid(alpha=0.3)
    mlflow.log_figure(fig, 'multihorizon_consistency.png')
    plt.show()

## 9. VSN (Variable Selection Network) 가중치
결정적: `tft.interpret_output(...)` static / encoder / decoder 변수 가중치 plot + MLflow 등록.

In [ ]:
raw_predictions = best_tft.predict(val_dl, mode='raw', return_x=True)
interpretation = best_tft.interpret_output(raw_predictions.output, reduction='sum')
figs = best_tft.plot_interpretation(interpretation)

with mlflow.start_run(run_name='tft_vsn'):
    for name, fig in (figs.items() if isinstance(figs, dict) else []):
        mlflow.log_figure(fig, f'vsn_{name}.png')
        plt.show()

## 10. Attention heatmap (case study)
결정적: 대표 SC 3개 sample의 attention(decoder × encoder lookback) heatmap → MLflow.

In [ ]:
attn = raw_predictions.output['attention']  # (n_samples, decoder_len, n_heads, encoder_len)
n_samples = attn.shape[0]
case_idx = sorted(set([0, n_samples // 2, n_samples - 1]))

with mlflow.start_run(run_name='tft_attention_case_study'):
    for idx in case_idx:
        attn_avg = attn[idx].mean(dim=1).cpu().numpy()  # (decoder_len, encoder_len), heads 평균
        fig, ax = plt.subplots(figsize=(10, 3))
        sns.heatmap(attn_avg, ax=ax, cmap='viridis', cbar_kws={'label': 'attention'})
        ax.set_xlabel('encoder lookback (오래된 ← → 최근)'); ax.set_ylabel('forecast horizon h')
        ax.set_title(f'Attention heatmap — sample idx={idx}')
        mlflow.log_figure(fig, f'attention_sample_{idx}.png')
        plt.show()

## 11. PoC 종합 비교 + AC 체크
결정적: 비교 표 + Acceptance Criteria 자동 체크 (t+1 LightGBM ±2%p, t+2~t+8 Naive ≤).

In [ ]:
print('=== horizon별 비교 ===')
print(compare.round(4).to_markdown())

if 'wape_lgb' in compare.columns and pd.notna(compare.loc[1, 'wape_lgb']):
    pass_t1 = abs(compare.loc[1, 'wape_tft'] - compare.loc[1, 'wape_lgb']) <= 0.02
    t1_msg = f'TFT={compare.loc[1, "wape_tft"]:.4f} vs LightGBM={compare.loc[1, "wape_lgb"]:.4f}'
else:
    pass_t1 = None
    t1_msg = 'LightGBM 베이스라인 미회수'

h_multi = compare.loc[2:DECODER_LEN]
pass_multi = bool((h_multi['wape_tft'] <= h_multi['wape_naive']).all())

verdict = 'PASS' if (pass_t1 and pass_multi) else ('PARTIAL' if pass_multi else 'FAIL')
print(f'\n=== PoC Verdict: {verdict} ===')
print(f'  pass_t1 (LightGBM ±2%p): {pass_t1}  [{t1_msg}]')
print(f'  pass_multi (Naive baseline ≤ for t+2..t+8): {pass_multi}')
if not pass_multi:
    fail_h = h_multi[h_multi['wape_tft'] > h_multi['wape_naive']].index.tolist()
    print(f'    └ Naive에 진 horizon: {fail_h}')

with mlflow.start_run(run_name='poc_verdict'):
    mlflow.log_metric('pass_t1', float(pass_t1) if pass_t1 is not None else float('nan'))
    mlflow.log_metric('pass_multi', float(pass_multi))
    mlflow.log_text(verdict, 'verdict.txt')

# spec AC 체크박스 갱신은 specs/train-sc-tft-multistep.md 직접 수정 (PoC 종료 시)